In [2]:
# Importing necessary libraries
import requests
import time
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta
import os
from dotenv import load_dotenv
import networkx as nx
from collections import defaultdict
import numpy as np

load_dotenv()

# Loading the GFW API Token from .env file
API_TOKEN = os.getenv("GFW_API_TOKEN")

# Setting the correct headers for the API Requests for extracting the data
HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

# Setting the API link paths
BASE_EVENTS = "https://gateway.api.globalfishingwatch.org/v3/events"

In [ ]:
# Defining the function to generate month ranges given a start and end date
def generate_month_ranges(start, end):
    ranges = [] # Initialising the list to store the month ranges
    current = start # Setting the current var to the start date
    
    while current < end: # Iterating while the current var is lesser than the end date
        next_month = current + relativedelta(months=1) # Calculting the start of next month
        ranges.append((current.strftime("%Y-%m-%d"), next_month.strftime("%Y-%m-%d"))) # Appending a new month range to the list
        current = next_month # Updating the cur var to the start of next month
    
    return ranges # Returning the month ranges calculated above

# Defining a function to download the port events of give month's end and start date
def download_events_month(start_date, end_date, out_file):

    all_entries = [] # List to store the events for the given month's start and end dates
    offset = 0 # Initialising the offset
    limit = 100000 # Initialising the var to limit the no. of entries retrieved in every API call

    # Iterating while all entries haven't been retrieved for the given start and end dates
    while True:
        print(f"Fetching offset {offset}")

        # Setting the limit and offset params for the API request
        params = {
            "limit": limit,
            "offset": offset
        }

        # Setting the payload with the filters of the data points as needed for constructing the edge list
        payload = {
            "datasets": ["public-global-port-visits-events:latest"],
            "types": ["PORT_VISIT"],
            "vesselTypes": ["CARGO", "CONTAINER"],
            "startDate": start_date,
            "endDate": end_date
        }

        # Sending a POST request to the above defined API link to retrieve the filtered data
        response = requests.post(
            BASE_EVENTS,
            params=params,
            json=payload,
            headers=HEADERS
        )

        # Converting the obtained response to json format
        data = response.json()

        # Extracting the port events from the "entries" field of the obtained json response
        entries = data.get("entries", [])

        # Checking to see if entries is blank
        if not entries:
            break

        # Appending the data for this set of limit and offset params to the global list of entries for this month
        all_entries.extend(entries)

        # Printing how much data has been obtained for debugging
        print(f"Collected: {len(all_entries) * 100 / data.get('total')}")

        # Stopping if its the last batch
        if len(entries) < limit:
            break

        # Updating the offset
        offset += limit
        time.sleep(1)  # Forcing a time delay of 1 sec between API calls to avoid rate limit errors

    # Initialising a list to store the filtered entry rows having only necessary info
    processed = []

    # Iterating over all above obtained entries
    for e in all_entries:
        try:
            # Appending a new row to the df acc. to whatever data we deem to be important from the obtained entries
            processed.append({
                "ship_id": e["vessel"]["ssvid"],
                "ship_type": e["vessel"]["type"],
                "port_id": e["port_visit"]["startAnchorage"]["id"],
                "port_name": e["port_visit"]["startAnchorage"]["name"],
                "lat": e["port_visit"]["startAnchorage"]["lat"],
                "lon": e["port_visit"]["startAnchorage"]["lon"],
                "entry_time": e["start"],
                "exit_time": e["end"],
                "duration_hrs": e["port_visit"]["durationHrs"]
            })
        except Exception as err:
            continue  # skip malformed entries

    # Converting the obtained entries to DataFrame (flatten important fields)
    df = pd.DataFrame(processed)
    df.to_csv(out_file, index=False) # Saving the above created df as a csv file for later use

    print(f"Saved {len(df)} rows → {out_file}")

# Defining a function to download the port events (PORT ENTRY, PORT EXIT, PORT VISIT) for an entire year
def download_events_year(start_date, end_date, out_dir):

    # Constructing the output dir where each month's port events will be saved as CSV files after retrieval and extraction
    os.makedirs(out_dir, exist_ok=True)
    
    month_ranges = generate_month_ranges(start_date, end_date) # Generating the month ranges for easier data retrieval by segmenting a year into 12 months
    
    files = [] # List to store the file names for each month's data
    
    # Iterating over each month of the given year
    for start, end in month_ranges:
        print(f"Events: {start} → {end}")
        
        # Defining the filename for this month's data
        filename = f"{out_dir}/events_{start}.csv"
        download_events_month(start, end, filename) # Downloading this month's data and extracting relevant info and saving to the above defined file
        
        files.append(filename) # Appending this month's file name to the list of filenames for the year
    
    return files # Returning the list of files for the data corresponding to each month for the given year

# Defining a function to merge the data of all csv files for months of a year into a single CSV file
def merge_csv(files, output_file):
    df_list = [] # List to store the df for each month's data for the given year's files
    
    for f in files: # Iterating over the given year's month data csv files
        try:
            df = pd.read_csv(f) # Reading the csv data into the dataframe
            df_list.append(df) # Appending the df of this month to the above defined list
        except:
            print(f"Skipping {f}")
    
    # Concatenating all the dataframes into a single dataframe and saving as a single csv file for the year
    combined = pd.concat(df_list, ignore_index=True)
    combined.to_csv(output_file, index=False)
    
    print(f"Saved merged file: {output_file}")

In [ ]:
# Defining the start and end dates for the year 2015
start_1 = datetime(2015, 1, 1)
end_1   = datetime(2016, 1, 1)

# Defining the start and end dates for the year 2025
start_2 = datetime(2025, 1, 1)
end_2   = datetime(2026, 1, 1)

In [ ]:
# Obtaining the file names for the months data for the year 2015
files_ev_2015 = download_events_year(start_1, end_1, "data/events_2015")

# Merging the data files for the year 2015
merge_csv(files_ev_2015, "data/events_2015_full.csv")

Events: 2015-01-01 → 2015-02-01
Fetching offset 0
Collected: 54.17499607231279
Fetching offset 100000
Collected: 100.0
Saved 184587 rows → data/events_2015/events_2015-01-01.csv
Events: 2015-02-01 → 2015-03-01
Fetching offset 0
Collected: 62.984984379723876
Fetching offset 100000
Collected: 100.0
Saved 158768 rows → data/events_2015/events_2015-02-01.csv
Events: 2015-03-01 → 2015-04-01
Fetching offset 0
Collected: 54.24729170396168
Fetching offset 100000
Collected: 100.0
Saved 184341 rows → data/events_2015/events_2015-03-01.csv
Events: 2015-04-01 → 2015-05-01
Fetching offset 0
Collected: 54.039740825403
Fetching offset 100000
Collected: 100.0
Saved 185049 rows → data/events_2015/events_2015-04-01.csv
Events: 2015-05-01 → 2015-06-01
Fetching offset 0
Collected: 53.38999791779008
Fetching offset 100000
Collected: 100.0
Saved 187301 rows → data/events_2015/events_2015-05-01.csv
Events: 2015-06-01 → 2015-07-01
Fetching offset 0
Collected: 52.784933068704866
Fetching offset 100000
Collecte

In [ ]:
# Obtaining the file names for the months data for the year 2025
files_ev_2025 = download_events_year(start_2, end_2, "data/events_2025")

# Merging the data files for the year 2025
merge_csv(files_ev_2025, "data/events_2025_full.csv")

Events: 2025-01-01 → 2025-02-01
Fetching offset 0
Collected: 51.252351201611376
Fetching offset 100000
Collected: 100.0
Saved 195113 rows → data/events_2025/events_2025-01-01.csv
Events: 2025-02-01 → 2025-03-01
Fetching offset 0
Collected: 55.402582868413326
Fetching offset 100000
Collected: 100.0
Saved 180497 rows → data/events_2025/events_2025-02-01.csv
Events: 2025-03-01 → 2025-04-01
Fetching offset 0
Collected: 47.76073780787765
Fetching offset 100000
Collected: 95.5214756157553
Fetching offset 200000
Collected: 100.0
Saved 209377 rows → data/events_2025/events_2025-03-01.csv
Events: 2025-04-01 → 2025-05-01
Fetching offset 0
Collected: 48.658248789626064
Fetching offset 100000
Collected: 97.31649757925213
Fetching offset 200000
Collected: 100.0
Saved 205515 rows → data/events_2025/events_2025-04-01.csv
Events: 2025-05-01 → 2025-06-01
Fetching offset 0
Collected: 47.017890307261915
Fetching offset 100000
Collected: 94.03578061452383
Fetching offset 200000
Collected: 100.0
Saved 2126

In [3]:
# Reading the data from the csv files created above into dataframes
df_2015 = pd.read_csv("../Data/events_2015_full.csv")
df_2025 = pd.read_csv("../Data/events_2025_full.csv")

# Converting the timestamps to datetime format for each row
df_2015["entry_time"] = pd.to_datetime(df_2015["entry_time"])
df_2015["exit_time"] = pd.to_datetime(df_2015["exit_time"])
df_2025["entry_time"] = pd.to_datetime(df_2025["entry_time"])
df_2025["exit_time"] = pd.to_datetime(df_2025["exit_time"])

# Sorting the rows in each dataframe by ship id and port visit event entry time
df_2015 = df_2015.sort_values(["ship_id", "entry_time"])
df_2025 = df_2025.sort_values(["ship_id", "entry_time"])

MIN_TIME_GAP_HRS = 1 # Setting the threshold for accurate port visit events

# Grouping rows in both dataframes by ship id and defining a new gap col based on the min time gap threshold defined above
df_2015["gap"] = df_2015.groupby("ship_id")["entry_time"].diff().dt.total_seconds() / 3600
df_2025["gap"] = df_2025.groupby("ship_id")["entry_time"].diff().dt.total_seconds() / 3600

# Only keeping rows where consecutive entries have at least 1 hr gap between entry times
df_2015 = df_2015[(df_2015["gap"].isna()) | (df_2015["gap"] < 1000)]
df_2025 = df_2025[(df_2025["gap"].isna()) | (df_2025["gap"] < 1000)]

In [4]:
# Defining a function to build ship sequences given dataframe
def build_ship_sequences(df):

    # Dictionary to store the ship sequences
    ship_sequences = {}

    # Iterating over ship ids after grouping all rows of the given dataframe by the ship id col
    for ship_id, group in df.groupby("ship_id"):
        group = group.sort_values("entry_time") # Sorting the group of entries per ship id by entry time
        
        ports = group["port_id"].tolist() # Extracting the list of ports visited by each ship
        
        # Storing the cleaned list of ports visited by each ship where duplicate port entries are removed
        cleaned_ports = []
        for p in ports:
            if len(cleaned_ports) == 0 or cleaned_ports[-1] != p:
                cleaned_ports.append(p)
        
        # If the port visit list for this ship has more than one entry, we can save it in the above defined dictionary
        if len(cleaned_ports) > 1:
            ship_sequences[ship_id] = cleaned_ports

    return ship_sequences # Returning the constructed ship-wise port visit events all throughout the year

# Extracting the ship-wie port visit events all throughout the year for 2015 and 2025
ship_sequences_2015 = build_ship_sequences(df_2015)
ship_sequences_2025 = build_ship_sequences(df_2025)

# Printing the total number of unique ships found in both year's data
print("Total ships:", len(ship_sequences_2015))
print("Total ships:", len(ship_sequences_2025))

Total ships: 31936
Total ships: 38689


In [5]:
# Defining a function to build a valid graph out of the given list of ship sequences
def build_graph(ship_sequences):

    # Defining an empty dictionary to save weights (no. of trips) from a directed route from a port to another port
    edge_weights = defaultdict(int)

    # Iterating over each ship's port visit sequence
    for ship_id, ports in ship_sequences.items():
        for i in range(len(ports) - 1):

            # Extracting the edge endpoints for creating/adding the edge
            A = ports[i] 
            B = ports[i+1]

            if A != B: # If the start and end ports of the edge are not the same, we add this trip of this ship as a weight to the graph
                edge_weights[(A, B)] += 1

    return edge_weights # Returning the above constructed graph

# Constructing the graphs for the 2015 and 2025 ship port visit sequence data we extracted before
edge_weights_2015 = build_graph(ship_sequences_2015)
edge_weights_2025 = build_graph(ship_sequences_2025)

In [6]:
# Defining a function to create a NetworkX graph from the given directed edge weight dict
def create_networkx_graph(edge_weights):
    G = nx.DiGraph() # Initialising an empty directed graph object

    # Iterating over each edge and its corresponding weight in the given directed edge weight dict
    for (A, B), w in edge_weights.items():
        G.add_edge(A, B, weight=w) # Adding this edge with its corresponding weight to the above defined directed graph object

    return G # Returning the constructed weighted directed graph

# Creating the weighted directed graph from the above created directed edge weight dict
G_2015 = create_networkx_graph(edge_weights_2015)
G_2025 = create_networkx_graph(edge_weights_2025)

# Printing basic stats for the graphs created for debugging
print("Nodes in 2015 graph:", G_2015.number_of_nodes())
print("Edges in 2015 graph:", G_2015.number_of_edges())
print("Nodes in 2025 graph:", G_2025.number_of_nodes())
print("Edges in 2025 graph:", G_2025.number_of_edges())

Nodes in 2015 graph: 7097
Edges in 2015 graph: 188545
Nodes in 2025 graph: 7717
Edges in 2025 graph: 182300


In [7]:
edges_2015 = [] # List to save the edge details for 2015 data
edges_2025 = [] # List to save the edge details for 2025 data

# Iterating over edge endpoints and weights for the 2015 weighted directed graph and appending the edge info to the respective list
for u, v, data in G_2015.edges(data=True):
    edges_2015.append({
        "source": u,
        "target": v,
        "weight": data["weight"]
    })

# Iterating over edge endpoints and weights for the 2025 weighted directed graph and appending the edge info to the respective list
for u, v, data in G_2025.edges(data=True):
    edges_2025.append({
        "source": u,
        "target": v,
        "weight": data["weight"]
    })

# Creating a dataframe from the above created list for the edge info of 2015 data and saving it as a weighted edge list for analysis
edges_df_2015 = pd.DataFrame(edges_2015)
edges_df_2015.to_csv("../EdgeList/shipping_network_2015.csv", index=False)

# Creating a dataframe from the above created list for the edge info of 2025 data and saving it as a weighted edge list for analysis
edges_df_2025 = pd.DataFrame(edges_2025)
edges_df_2025.to_csv("../EdgeList/shipping_network_2025.csv", index=False)